# <font color="steelblue">27. Intervalo de confianza para una proporción: el margen de error</font>

*Unidad 5. Conceptos básicos de inferencia estadística* · Sesión teórica 27 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`sondeos.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["sondeos.csv"]
PREGUNTAS = ["q_P27_1.json", "q_P27_2.json", "q_P27_3.json", "q_P27_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«Error muestral: ±3,1 puntos para un nivel de confianza del 95 %». Esa línea de la ficha técnica es la clave para leer cualquier encuesta. Con ella sabrás cuándo una diferencia es noticia y cuándo es un **empate técnico**, y por qué los resultados de un subgrupo (los jóvenes, una provincia) son mucho menos precisos que el total.

**Qué necesitas saber antes**: los cuadernos 24 y 26.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular el margen de error y el intervalo de confianza de una proporción.
2. Interpretar correctamente el nivel de confianza.
3. Decidir cuándo dos resultados de una encuesta son un empate técnico.
4. Calcular el margen de error de un subgrupo.

## <font color='steelblue'>Contenidos</font>

1. El margen de error
2. Qué significa «95 % de confianza»
3. Empate técnico
4. Los subgrupos tienen más error

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. El margen de error</font>

El **intervalo de confianza** al 95 % para una proporción es:

$$\hat{p} \pm 1{,}96 \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

La parte de la derecha es el **margen de error**. Las fichas técnicas suelen dar el margen «en el peor caso», con $p = 0{,}5$, que es el mayor posible.

In [ ]:
def margen(p, n, confianza=0.95):
    z = stats.norm.ppf(1 - (1 - confianza) / 2)     # 1,96 para el 95 %
    return z * np.sqrt(p * (1 - p) / n)

def intervalo(p, n, confianza=0.95):
    e = margen(p, n, confianza)
    return p - e, p + e

print(f"Margen para n = 1.000, p = 0,5: ±{margen(0.5, 1000) * 100:.1f} puntos")
inf, sup = intervalo(0.34, 1000)
print(f"Partido con 34 % en 1.000 entrevistas: entre {inf:.1%} y {sup:.1%}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P27_1.json")

## <font color='steelblue'>2. Qué significa «95 % de confianza»</font>

El 95 % **no** es la probabilidad de que el valor real esté en *este* intervalo concreto. Significa que, si repitiéramos la encuesta muchas veces, el **95 % de los intervalos** así calculados contendría el valor real. Lo vemos con 100 encuestas simuladas:

In [ ]:
rng = np.random.default_rng(27)
p_real, n = 0.34, 1000
fallan = 0
for i in range(100):
    p_hat = rng.binomial(n, p_real) / n
    lo, hi = intervalo(p_hat, n)
    acierta = lo <= p_real <= hi
    fallan += not acierta
    plt.plot([lo * 100, hi * 100], [i, i], color="steelblue" if acierta else "firebrick")
plt.axvline(34, color="black")
plt.xlabel("% de voto a A")
plt.title(f"100 intervalos al 95 %: {fallan} no contienen el valor real (en rojo)")
plt.show()

Subir la confianza al 99 % ensancha el intervalo (z = 2,58). Es un compromiso entre **seguridad** y **precisión**.

In [ ]:
for c in [0.90, 0.95, 0.99]:
    print(f"Confianza {c:.0%}: margen ±{margen(0.5, 1000, c) * 100:.1f} puntos")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P27_2.json")

## <font color='steelblue'>3. Empate técnico</font>

In [ ]:
sondeos = pd.read_csv("sondeos.csv")
sondeos.tail()

Si los intervalos de dos candidaturas **se solapan claramente**, la encuesta no permite decir quién va delante: es un **empate técnico**. (La forma exacta de comparar dos porcentajes de la misma encuesta la verás en el cuaderno 30; el solapamiento es una regla práctica y algo conservadora.)

In [ ]:
ultimo = sondeos.iloc[-1]
n = ultimo["tamano_muestra"]
for cand in ["A", "B"]:
    p = ultimo[f"pct_{cand}"] / 100
    lo, hi = intervalo(p, n)
    print(f"{cand}: {p:.1%}  (entre {lo:.1%} y {hi:.1%})")

In [ ]:
sondeos["margen"] = margen(0.5, sondeos["tamano_muestra"]) * 100
plt.errorbar(range(len(sondeos)), sondeos["pct_A"], yerr=sondeos["margen"], fmt="o", label="A", alpha=0.7)
plt.errorbar(np.arange(len(sondeos)) + 0.3, sondeos["pct_B"], yerr=sondeos["margen"], fmt="o", label="B", alpha=0.7)
plt.xlabel("Sondeo (en orden de fecha)")
plt.ylabel("% de voto")
plt.legend()
plt.title("Sondeos simulados con su margen de error")
plt.show()

**Cómo leerlo**: casi todos los sondeos tienen las barras de A y B solapadas. Un titular como «B adelanta a A» basado en uno solo de ellos sería engañoso; lo correcto es hablar de empate técnico o mirar la **media de muchos sondeos**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P27_3.json")

## <font color='steelblue'>4. Los subgrupos tienen más error</font>

Si la encuesta tiene 1.000 entrevistas pero solo 150 son jóvenes, el margen para los jóvenes se calcula con **n = 150**, y es mucho mayor. Las noticias que destacan «lo que opinan los jóvenes» a partir de una encuesta general deben decirlo.

In [ ]:
for grupo, n_g in [("Total", 1000), ("Mayores de 65", 260), ("De 18 a 29", 150), ("Castellón", 60)]:
    print(f"{grupo:>15} (n = {n_g:>4}): ±{margen(0.5, n_g) * 100:.1f} puntos")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P27_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Calcula el intervalo al 95 % para un partido que obtiene el 12 % en una encuesta de 800 personas.

<details><summary>Ver solución</summary>

```python
intervalo(0.12, 800)
```
</details>

### **Ejercicio 2** *(sección 2)*

Repite la simulación de los 100 intervalos al 90 %. ¿Cuántos fallan ahora, aproximadamente?

<details><summary>Ver idea clave</summary>

Cambia `intervalo(p_hat, n)` por `intervalo(p_hat, n, 0.90)`: deberían fallar unos 10 de cada 100.
</details>

### **Ejercicio 3** *(sección 3)*

¿En cuántos sondeos la diferencia entre A y B es mayor que dos veces el margen (es decir, no se solapan los intervalos)?

<details><summary>Ver solución</summary>

```python
((sondeos["pct_A"] - sondeos["pct_B"]).abs() > 2 * sondeos["margen"]).sum()
```
</details>

### **Ejercicio 4** *(sección 3)*

Calcula la media de todos los sondeos para A y B y su margen aproximado, sumando todas las muestras. ¿Sigue siendo un empate?

<details><summary>Ver solución</summary>

```python
n_tot = sondeos["tamano_muestra"].sum()
pa = np.average(sondeos["pct_A"], weights=sondeos["tamano_muestra"]) / 100
pb = np.average(sondeos["pct_B"], weights=sondeos["tamano_muestra"]) / 100
print(pa, pb, margen(0.5, n_tot))
```

Promediar sondeos reduce mucho el margen, siempre que los sondeos no compartan el mismo sesgo.
</details>

### **Ejercicio 5** *(sección 4)*

Redacta la frase de ficha técnica de una encuesta de 1.600 entrevistas al 95 %.

<details><summary>Ver solución</summary>

```python
print(f"Error muestral: ±{margen(0.5, 1600) * 100:.1f} puntos para un nivel de confianza del 95 % y p = q = 0,5")
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- **Intervalo al 95 %** para una proporción: $\hat{p} \pm 1{,}96\sqrt{\hat{p}(1-\hat{p})/n}$; la parte ± es el **margen de error**.
- La ficha técnica da el margen con **p = 0,5**, el mayor posible.
- «95 % de confianza» es una propiedad del **método**: acierta en el 95 % de las muestras posibles.
- Si las diferencias son menores que el margen, es un **empate técnico**.
- Los **subgrupos** tienen menos casos y **más margen** de error.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Intervalo de confianza** | Rango de valores plausibles para un parámetro. |
| **Margen de error** | Semiamplitud del intervalo de confianza. |
| **Nivel de confianza** | Proporción de intervalos que contendrían el parámetro si se repitiera el muestreo. |
| **Empate técnico** | Situación en la que la diferencia entre dos resultados es menor que el error de la encuesta. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| El margen sale en tanto por uno (0,031) | Falta multiplicar por 100 | Multiplica por 100 para expresarlo en puntos |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 28 harás intervalos para medias y calcularás cuántas entrevistas necesita una encuesta.

**Siguiente cuaderno**: [28. Intervalo de confianza para una media y tamaño de muestra](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P28_intervalo_para_una_media_y_tamano_de_muestra.ipynb)